# 배터리 수명 예측: Batch 1 전처리

로컬 Jupyter에서 위에서 아래 순서로 실행합니다. 필요한 패키지는 `numpy`, `pandas`, `h5py`입니다.

- Batch 1의 46개 셀을 모두 사용합니다. 결측 셀 제외나 결측값 대체는 수행하지 않습니다.
- 원본 MATLAB v7.3 파일에서 필요한 데이터만 읽습니다.
- `delta_q_min = min(Qdlin_100 - Qdlin_10)`
- `QD_change_10_100 = QDischarge_100 - QDischarge_10`
- `mean_Tavg = mean(Tavg_2 ... Tavg_100)`: 기존 EDA와 동일하게, 0으로 기록된 첫 회차를 제외합니다.
- 100회차 이후 데이터는 입력 특징 계산에 사용하지 않습니다.
- 표준화와 학습/검증 분리는 이후 모델링 단계에서 진행합니다.


In [10]:
from pathlib import Path
import sys

import h5py
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "Data").is_dir() and (ROOT.parent / "Data").is_dir():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "Data"

BATCH_LABEL = "Batch 2"
SOURCE_NAME = "2018-02-20_batchdata_updated_struct_errorcorrect.mat"
FEATURE_COLUMNS = ["delta_q_min", "QD_change_10_100", "mean_Tavg"]
TARGET_COLUMN = "cycle_life"
WRITE_CSV = True
OUTPUT_PATH = ROOT / "Data" / "batch2_features.csv"

if not (DATA_DIR / SOURCE_NAME).is_file():
    raise FileNotFoundError(f"원본 파일이 없습니다: {DATA_DIR / SOURCE_NAME}")
print(f"Python: {sys.version.split()[0]}")
print(f"원본 파일: {DATA_DIR / SOURCE_NAME}")


Python: 3.12.14
원본 파일: /Users/uk/Desktop/archive/Data/2018-02-20_batchdata_updated_struct_errorcorrect.mat


## 1. 셀별 특징 추출

확인된 Batch 1 구조를 사용하여 각 셀의 특징 3개와 수명값을 바로 계산합니다. `summary.cycle`에서 실제 10·100회차 위치를 찾으며, 결측 검사에 따른 셀 제외나 제외 사유 기록은 하지 않습니다.


In [11]:
def referenced_object(handle, field, index):
    """MATLAB 셀별 객체 참조를 하나만 읽습니다."""
    if field.ndim != 2 or 1 not in field.shape:
        raise ValueError(f"지원하지 않는 참조 배열: {field.name}, {field.shape}")
    ref = field[index, 0] if field.shape[1] == 1 else field[0, index]
    if not ref:
        raise ValueError(f"비어 있는 객체 참조: {field.name}, index={index}")
    return handle[ref]


def read_vector(dataset):
    return np.asarray(dataset[()], dtype=float).reshape(-1)


def read_text(dataset):
    if dataset.attrs.get("MATLAB_class") != b"char":
        raise ValueError(f"문자열 형식이 아닙니다: {dataset.name}")
    return "".join(chr(int(value)) for value in dataset[()].reshape(-1)).strip()


def extract_cell(handle, batch, batch_label, index, source_name):
    """배터리 셀 하나에서 입력 특징 3개와 수명·메타데이터를 추출합니다."""
    summary = referenced_object(handle, batch["summary"], index)
    curves = referenced_object(handle, batch["cycles"], index)
    cycle_numbers = read_vector(summary["cycle"])
    positions = {int(number): position for position, number in enumerate(cycle_numbers)}
    pos10 = positions[10]
    pos100 = positions[100]

    # 방전 용량은 필요한 10회차와 100회차 값만 읽습니다.
    qd10 = float(summary["QDischarge"][0, pos10])
    qd100 = float(summary["QDischarge"][0, pos100])
    temp_early = np.asarray(summary["Tavg"][0, :pos100 + 1], dtype=float)
    temp_positions = [positions[number] for number in range(2, 101)]

    q10 = read_vector(referenced_object(handle, curves["Qdlin"], pos10))
    q100 = read_vector(referenced_object(handle, curves["Qdlin"], pos100))
    delta_q = q100 - q10

    return {
        "batch_id": batch_label,
        "cell_id": f"b{batch_label.split()[-1]}c{index}",
        "source_cell_index": index,
        "source_file": source_name,
        "charging_policy": read_text(
            referenced_object(handle, batch["policy_readable"], index)
        ),
        "cycle_life": float(
            read_vector(referenced_object(handle, batch["cycle_life"], index)).item()
        ),
        "delta_q_min": float(delta_q.min()),
        "QD_change_10_100": qd100 - qd10,
        "mean_Tavg": float(temp_early[temp_positions].mean()),
    }


In [14]:
with h5py.File(DATA_DIR / SOURCE_NAME, "r") as handle:
    batch = handle["batch"]
    n_cells = batch["summary"].size
    records = [
        extract_cell(handle, batch, BATCH_LABEL, index, SOURCE_NAME)
        for index in range(n_cells)
    ]

features_df = pd.DataFrame(records)
batch1_df = features_df.copy()
print(f"Batch 1: {len(features_df)}개 셀 처리 완료")

features_df = pd.DataFrame(records)
features_df = features_df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)


Batch 1: 47개 셀 처리 완료


## 2. 최종 결과 확인

모든 셀을 그대로 유지한 `features_df`에서 입력 특징과 수명값의 유한 여부를 마지막에 한 번 확인합니다. 문제가 있으면 셀을 제외하지 않고 실행을 중단합니다.

`charging_policy`는 이후 충전 프로토콜 단위 학습/검증 분리에 사용하는 메타데이터입니다. 식별 정보와 파일 정보도 모델 입력에서 제외하고, `FEATURE_COLUMNS`의 3개 변수만 입력으로 사용합니다.


In [16]:
assert np.isfinite(
    features_df[FEATURE_COLUMNS + [TARGET_COLUMN]].to_numpy()
).all(), "입력 변수 또는 수명값에 결측값·무한대가 있습니다."

display(features_df.head())
print(f"셀 수: {len(features_df)}")
print(f"충전 프로토콜 수: {features_df['charging_policy'].nunique()}")
print("입력 특징:", FEATURE_COLUMNS)
print("최종 유한값 확인 완료")


,batch_id,cell_id,source_cell_index,source_file,charging_policy,cycle_life,delta_q_min,QD_change_10_100,mean_Tavg
0,Batch 2,b2c0,0,2018-02-20_batchdata_updated_struct_errorcorre...,5.2C(58%)-4C,477.0,-0.055061,0.001554,31.808471
1,Batch 2,b2c1,1,2018-02-20_batchdata_updated_struct_errorcorre...,5.6C(26%)-4.5C,491.0,-0.066115,-0.006769,33.600998
2,Batch 2,b2c2,2,2018-02-20_batchdata_updated_struct_errorcorre...,5.2C(50%)-4.25C,424.0,-0.052718,-0.005760,33.175661
3,Batch 2,b2c3,3,2018-02-20_batchdata_updated_struct_errorcorre...,4.65C(44%)-5C,499.0,-0.055868,0.001901,33.938058
4,Batch 2,b2c4,4,2018-02-20_batchdata_updated_struct_errorcorre...,4.65C(69%)-6C,444.0,-0.070297,-0.009355,33.048209


셀 수: 39
충전 프로토콜 수: 12
입력 특징: ['delta_q_min', 'QD_change_10_100', 'mean_Tavg']
최종 유한값 확인 완료


## 3. Batch 1 CSV 저장

기본값은 `WRITE_CSV = False`입니다. True로 변경하고 실행하면 `processed/batch1_features.csv`에 저장합니다. 같은 이름의 파일이 있으면 덮어씁니다.

기존에 생성한 전체 배치 CSV와 제외 내역 파일은 이 노트북에서 사용하지 않습니다.


In [17]:
if WRITE_CSV:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    features_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    print(f"저장 완료: {OUTPUT_PATH}")
else:
    print("CSV 저장은 꺼져 있습니다. Batch 1 결과는 features_df에 있습니다.")


저장 완료: /Users/uk/Desktop/archive/Data/batch2_features.csv
